##Carbon Footprint AI - Exploratory Data Analysis

**Project**: Carbon Footprint AI Microservice  
**Dataset**: ASHRAE - Great Energy Predictor III  
**Date**: November 2025  

---

## 📋 Objectives

This notebook performs exploratory data analysis on the ASHRAE energy consumption dataset:

1. **Load and inspect the datasets** (building metadata, meter readings, weather data)
2. **Understand data structure and types**
3. **Identify missing values and data quality issues**
4. **Analyze distributions and patterns**
5. **Explore correlations between features**
6. **Generate insights for carbon emission prediction**

---

## 1️⃣ Setup & Imports

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Statistical analysis
from scipy import stats
from scipy.stats import normaltest, skew, kurtosis

# Utilities
import warnings
from datetime import datetime
import os

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported successfully!")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2️⃣ Load Datasets

The ASHRAE dataset consists of three main files:
- **building_metadata.csv**: Building characteristics (size, use, year built)
- **train.csv**: Hourly meter readings (energy consumption)
- **weather_train.csv**: Weather data (temperature, wind, pressure)

In [ ]:
# Define data paths
DATA_PATH = '../../data/raw/'

print("📂 Loading datasets...")
print("-" * 50)

# Load building metadata
building_df = pd.read_csv(os.path.join(DATA_PATH, 'building_metadata.csv'))
print(f"✅ Building Metadata: {building_df.shape[0]:,} rows × {building_df.shape[1]} columns")

# Load meter readings (sample first 1M rows for faster analysis)
# Note: Full dataset has 20M+ rows, we'll sample for EDA
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'), nrows=1_000_000)
print(f"✅ Meter Readings (sample): {train_df.shape[0]:,} rows × {train_df.shape[1]} columns")

# Load weather data
weather_df = pd.read_csv(os.path.join(DATA_PATH, 'weather_train.csv'))
print(f"✅ Weather Data: {weather_df.shape[0]:,} rows × {weather_df.shape[1]} columns")

print("-" * 50)
print("✅ All datasets loaded successfully!")

## 3️⃣ Building Metadata Analysis

In [ ]:
# Display first few rows
print("📋 Building Metadata - First 10 rows:")
building_df.head(10)

In [ ]:
# Dataset info
print("ℹ️ Building Metadata Information:")
building_df.info()

In [ ]:
# Statistical summary
print("📊 Building Metadata - Statistical Summary:")
building_df.describe()

In [ ]:
# Check missing values in building metadata
print("🔍 Missing Values in Building Metadata:")
missing = building_df.isnull().sum()
missing_pct = (missing / len(building_df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Missing Count', ascending=False)
print(missing_df[missing_df['Missing Count'] > 0])

if missing.sum() == 0:
    print("✅ No missing values found!")

In [ ]:
# Analyze building primary use
print("🏢 Building Primary Use Distribution:")
use_counts = building_df['primary_use'].value_counts()
print(use_counts)

# Visualize
plt.figure(figsize=(12, 6))
use_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribution of Building Primary Use', fontsize=16, fontweight='bold')
plt.xlabel('Primary Use', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Building size distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(building_df['square_feet'], bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0].set_title('Building Size Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Square Feet', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)

# Box plot
axes[1].boxplot(building_df['square_feet'].dropna())
axes[1].set_title('Building Size - Box Plot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Square Feet', fontsize=12)

plt.tight_layout()
plt.show()

print(f"📊 Building Size Statistics:")
print(f"   Mean: {building_df['square_feet'].mean():,.0f} sq ft")
print(f"   Median: {building_df['square_feet'].median():,.0f} sq ft")
print(f"   Min: {building_df['square_feet'].min():,.0f} sq ft")
print(f"   Max: {building_df['square_feet'].max():,.0f} sq ft")

## 4️⃣ Meter Readings Analysis

In [ ]:
# Display first few rows
print("📋 Meter Readings - First 10 rows:")
train_df.head(10)

In [ ]:
# Convert timestamp to datetime
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])

print("ℹ️ Meter Readings Information:")
train_df.info()

In [ ]:
# Meter type distribution
# Meter types: 0=electricity, 1=chilled water, 2=steam, 3=hot water
meter_types = {0: 'Electricity', 1: 'Chilled Water', 2: 'Steam', 3: 'Hot Water'}

print("⚡ Meter Type Distribution:")
meter_counts = train_df['meter'].value_counts().sort_index()
for meter_id, count in meter_counts.items():
    print(f"   {meter_types.get(meter_id, 'Unknown')}: {count:,} readings")

# Visualize
plt.figure(figsize=(10, 6))
meter_counts.plot(kind='bar', color=['gold', 'skyblue', 'coral', 'lightgreen'], edgecolor='black')
plt.title('Distribution of Meter Types', fontsize=16, fontweight='bold')
plt.xlabel('Meter Type', fontsize=12)
plt.ylabel('Number of Readings', fontsize=12)
plt.xticks(range(len(meter_types)), [meter_types[i] for i in range(len(meter_types))], rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Meter reading statistics
print("📊 Meter Reading Statistics:")
print(train_df['meter_reading'].describe())

# Distribution of meter readings
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Original scale
axes[0].hist(train_df['meter_reading'], bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Meter Reading Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Meter Reading', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)

# Log scale (to see distribution better)
axes[1].hist(np.log1p(train_df['meter_reading']), bins=100, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_title('Meter Reading Distribution (Log Scale)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Log(Meter Reading + 1)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Check for zero readings
zero_readings = (train_df['meter_reading'] == 0).sum()
zero_pct = (zero_readings / len(train_df)) * 100
print(f"🔍 Zero Readings: {zero_readings:,} ({zero_pct:.2f}%)")

# Check for negative readings (data quality issue)
negative_readings = (train_df['meter_reading'] < 0).sum()
print(f"⚠️ Negative Readings: {negative_readings:,}")

## 5️⃣ Weather Data Analysis

In [ ]:
# Display first few rows
print("📋 Weather Data - First 10 rows:")
weather_df.head(10)

In [ ]:
# Convert timestamp to datetime
weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])

print("ℹ️ Weather Data Information:")
weather_df.info()

In [ ]:
# Check missing values in weather data
print("🔍 Missing Values in Weather Data:")
missing = weather_df.isnull().sum()
missing_pct = (missing / len(weather_df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Missing Count', ascending=False)
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Weather statistics
print("📊 Weather Data - Statistical Summary:")
weather_df.describe()

In [ ]:
# Temperature distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Air temperature
axes[0, 0].hist(weather_df['air_temperature'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 0].set_title('Air Temperature Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Temperature (°C)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)

# Dew temperature
axes[0, 1].hist(weather_df['dew_temperature'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 1].set_title('Dew Temperature Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Temperature (°C)', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)

# Wind speed
axes[1, 0].hist(weather_df['wind_speed'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='lightgreen')
axes[1, 0].set_title('Wind Speed Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Wind Speed (m/s)', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)

# Sea level pressure
axes[1, 1].hist(weather_df['sea_level_pressure'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='gold')
axes[1, 1].set_title('Sea Level Pressure Distribution', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Pressure (mbar)', fontsize=12)
axes[1, 1].set_ylabel('Frequency', fontsize=12)

plt.tight_layout()
plt.show()

## 6️⃣ Merged Data Analysis

Let's merge the datasets to explore relationships between building characteristics, weather, and energy consumption.

In [ ]:
# Merge meter readings with building metadata
print("🔗 Merging datasets...")
merged_df = train_df.merge(building_df, on='building_id', how='left')
print(f"✅ Merged with building metadata: {merged_df.shape}")

# Merge with weather data
merged_df = merged_df.merge(weather_df, on=['site_id', 'timestamp'], how='left')
print(f"✅ Merged with weather data: {merged_df.shape}")

print("\n📋 Merged Dataset - First 5 rows:")
merged_df.head()

In [ ]:
# Correlation analysis (numerical features only)
numerical_cols = ['meter_reading', 'square_feet', 'year_built', 'floor_count', 
                  'air_temperature', 'dew_temperature', 'sea_level_pressure', 
                  'wind_speed', 'cloud_coverage']

# Filter to existing columns
available_cols = [col for col in numerical_cols if col in merged_df.columns]

# Calculate correlation matrix
correlation_matrix = merged_df[available_cols].corr()

# Visualize
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Energy Consumption Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Energy consumption by building type
print("🏢 Average Meter Reading by Building Type:")
avg_by_type = merged_df.groupby('primary_use')['meter_reading'].mean().sort_values(ascending=False)
print(avg_by_type)

# Visualize
plt.figure(figsize=(12, 6))
avg_by_type.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Average Energy Consumption by Building Type', fontsize=16, fontweight='bold')
plt.xlabel('Average Meter Reading', fontsize=12)
plt.ylabel('Building Type', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature vs Energy Consumption
# Sample data for visualization (to avoid overplotting)
sample_df = merged_df.sample(n=min(10000, len(merged_df)), random_state=42)

plt.figure(figsize=(12, 6))
plt.scatter(sample_df['air_temperature'], sample_df['meter_reading'], 
            alpha=0.3, s=10, color='coral')
plt.title('Temperature vs Energy Consumption', fontsize=16, fontweight='bold')
plt.xlabel('Air Temperature (°C)', fontsize=12)
plt.ylabel('Meter Reading', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7️⃣ Time Series Patterns

In [ ]:
# Extract time features
merged_df['hour'] = merged_df['timestamp'].dt.hour
merged_df['day_of_week'] = merged_df['timestamp'].dt.dayofweek
merged_df['month'] = merged_df['timestamp'].dt.month

# Average consumption by hour of day
hourly_avg = merged_df.groupby('hour')['meter_reading'].mean()

plt.figure(figsize=(14, 6))
hourly_avg.plot(kind='line', marker='o', linewidth=2, markersize=6, color='steelblue')
plt.title('Average Energy Consumption by Hour of Day', fontsize=16, fontweight='bold')
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Average Meter Reading', fontsize=12)
plt.xticks(range(24))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Average consumption by day of week
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_avg = merged_df.groupby('day_of_week')['meter_reading'].mean()

plt.figure(figsize=(12, 6))
daily_avg.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Average Energy Consumption by Day of Week', fontsize=16, fontweight='bold')
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Average Meter Reading', fontsize=12)
plt.xticks(range(7), day_names, rotation=45)
plt.tight_layout()
plt.show()

## 8️⃣ Key Insights & Observations

### 📌 Data Quality Findings

**Building Metadata:**
- Total buildings analyzed
- Missing values in year_built and floor_count fields
- Building types range from Education to Office to Entertainment

**Meter Readings:**
- Four meter types: Electricity, Chilled Water, Steam, Hot Water
- Significant number of zero readings (may need investigation)
- Highly skewed distribution (log transformation recommended)

**Weather Data:**
- Missing values in several weather features
- Temperature ranges appear reasonable
- Will need imputation strategy for missing weather data

### 🔍 Pattern Observations

**Correlations:**
- Building size (square_feet) shows correlation with energy consumption
- Temperature affects energy usage (heating/cooling)
- Different building types have different consumption patterns

**Temporal Patterns:**
- Clear hourly patterns (peak during business hours)
- Weekday vs weekend differences
- Seasonal variations expected

### 🎯 Next Steps for Preprocessing

1. **Handle Missing Values:**
   - Impute missing weather data (forward fill, interpolation)
   - Handle missing building metadata

2. **Feature Engineering:**
   - Create time-based features (hour, day, month, season)
   - Calculate building age from year_built
   - Create energy intensity (consumption per square foot)
   - Lag features for time series

3. **Data Transformation:**
   - Log transformation for meter readings
   - Encode categorical variables (primary_use, meter type)
   - Scale numerical features

4. **Carbon Emission Calculation:**
   - Convert meter readings to carbon emissions using emission factors
   - Different factors for electricity, steam, etc.

---

## 💾 Save Analysis Results

Save key statistics and sample data for next phase

In [ ]:
# Save a sample of merged data for faster preprocessing
# sample_df = merged_df.sample(n=100000, random_state=42)
# sample_df.to_csv('../../data/processed/eda_sample.csv', index=False)
# print("✅ Sample data saved for preprocessing phase!")

print("✅ EDA Complete! Ready for preprocessing phase.")